#### Scrape websites that are highly interactive and don't want a bot snooping around their systems

In [ ]:
import nodriver as uc
ny_url="https://www.glassdoor.com/Job/new-york-jobs-SRCH_IL.0,8_IC1132348.htm"
browser = await uc.start()
tab = await browser.get(ny_url)

await tab.sleep(5)

In [ ]:
# %pip install nodriver

In [ ]:
print(await tab.evaluate("document.title"))
print(await tab.evaluate("location.href"))

In [ ]:
button = await tab.find("Show more jobs", best_match=True, timeout=10)
await button.click()
await tab.sleep(3)

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
html = await tab.get_content()

search_url= await tab.evaluate("location.href")

soup = BeautifulSoup(html, "html.parser")

cards = soup.select('div[class*="JobCard_jobCardContainer"]')
print(len(cards), "job cards found")

In [ ]:
jobs = []

def get_text(card, selector):
    el = card.select_one(selector)
    return el.get_text(" ", strip=True) if el else ""

for card in cards[:3]:
    title_link = card.select_one('a[data-test="job-title"]')
    rating     = card.select_one('[class*="RatingSingleStar"][aria-valuenow]')
    snippet    = card.select_one('[data-test="descSnippet"]')
    tracking = card.select_one('a[data-test="job-link"]')
    description, skills = "", ""

    if snippet:
        for div in snippet.select('div'):
            t = div.get_text(" ", strip=True)
            if t.startswith("Skills:"):
                skills = t.replace("Skills:", "").strip()
            elif not description:
                description = t.replace("&hellip;", "…")

    jobs.append({
        "job_id":   title_link["id"].replace("job-title-", "") if title_link else "",
        "title":    title_link.get_text(strip=True) if title_link else "",
        "url":      title_link["href"] if title_link else "",
        "company":  get_text(card, '[class*="EmployerProfile_compactEmployerName"]'),
        "rating":   rating["aria-valuenow"] if rating else "",
        "location": get_text(card, '[data-test="emp-location"]'),
        "salary":   get_text(card, '[data-test="detailSalary"]'),
        "description": description,
        "skills":   skills,
        "search_page": search_url,
        "tracking_href": tracking["href"] if tracking else "",
    })

In [ ]:
import pandas as pd
df=pd.DataFrame(jobs)

In [ ]:
# https://www.glassdoor.com/job-listing/api/job-details?jobListingId=1010194519329&pageTypeEnum=SERP&queryString=%2Fpartner%2FjobListing.htm%3Fpos%3D101%26ao%3D1110586%26s%3D58%26guid%3D000001a19e544303a2aaaf2cf1489d99%26src%3DGD_JOB_AD%26t%3DSR%26vt%3Dw%26cs%3D1_c23696a5%26cb%3D1793657685326%26jobListingId%3D1010194519329%26cpc%3D334ABAF5D42DC775%26jrtk%3D5-yul1-0-1k6f58gp9l01d801-1f3dd7a19dc3cddb---6NYlbfkN0DCh93Oy_IdBQoQVGreB6yXTeV-fieZLZsdwvTkdN6YgqljTektTE9JJWbKVk8qajMfhR3VqaRbO9vUydyO_wQiAPYJhY7G1nOI9rHTyk4i2gWF5fHbH4-G6r8zDcwVR9XGAQPFyIt3Jn2XfaTjBr4vRqPgm8HjbGGShFt11wBO5K8CTm5ONbK8RgZf_jSPNkEcvOPvIE_l4q7ZisgS1jHyyBoskpqrl0ir55s2eP8AV2mSlyw0jgRxObr-YtNSdTAZmN5KbJma1htyw_7RCa13GtBc_NJfkJlGpiFQV6EDZ1YaAPT5-mnTv64EwF53UggaN9ssezF3jSH2XJDZ8wfkN0UulSDW77saZWa5A-kcOw1aP-zGmyhSg_uLz95E1l9Uxb3yHJjXerUDjlKPE0ZgXgPTJoUv4BW-40-v1n7PKPPhxSaaWPKPkrB_pkA-GQhuUwMrOiZz5p9S6LC_JLMqO5LA-QcgKljXKsRz5dREUh7H_nLAeY0rXKk8sG4-EU_2WmZ9RXaru0rFhEMMT0m391uWtyZ04hoX_b456dBxEHbYbi3pKZT85LZzhPwqZUKoKjagYvRJUp1me2PqgLhyoRkX2NJxBBJY7_5O5mum-zfLKOG4E5LDi06MQeha84rwxDQncSUAOzFtEEnof7uN7hcjvzYLk9FiaIwNK2SXHyg7otdck9poXHYN1TyUC4TYQRSgq1eVJLuzSX1PJkq7cRwHRzgaC-E3Q3gr4LvHzNH0sgKkkVg3x71pmDS_A-GrJVHeKKEeZ_-EU6gqhv2M9HInJfA7jnR6SkUWK57vWXcdqsup1EWQlixQVnun_BAR7WMYhWFSgy6K5TW44l7l18yLEucaeCbSkZqBSqBeqFVOUpS4CCm1PwLxaENIvJWDQBIE9McYE0iNlQpBZ2J9hauZpJHsZIc%253D&countryId=1

df['tracking_href'][0]

In [ ]:
## build the api url

from urllib.parse import urlencode


def job_api_url(job_id, tracking_href):
    params = {
        "jobListingId": job_id,
        "pageTypeEnum": "SERP",
        "queryString": tracking_href,
        "countryId": 1,
    }
    
    return "https://www.glassdoor.com/job-listing/api/job-details?" + urlencode(params)


In [ ]:
df['api_url']=df.apply(lambda row: job_api_url(row['job_id'], row['tracking_href']), axis=1)

In [ ]:
df.api_url[0]

In [ ]:
# 1. Fetch a small sample and save it to disk.

## Build a small piece of js as text. Go to this url, using my cookies and give me back what you find. This request will run inside the chrome window, which has already passed glassdoor's bot check. Python's request would get blocked.
#  

# fetch("https://www.glassdoor.com/job-listing/api/job-details?jobListingId=...", {credentials: 'include'}).then(r => r.text())
# js = "fetch(" + json.dumps(url) + ", {credentials: 'include'}).then(r => r.text())"


In [ ]:
import json

url = df["api_url"].iloc[0]          # first row's API link

js = f"fetch({json.dumps(url)}, {{credentials: 'include'}}).then(r => r.text())"
text = await tab.evaluate(js, await_promise=True)

data = json.loads(text)
print(data['employerName'])

print(data["isEasyApply"])
print(BeautifulSoup(data["jobDescription"], "html.parser").get_text(" ", strip=True))

In [ ]:
import json
import asyncio
import random

employer_names = []
easy_apply = []
descriptions = []

for url in df["api_url"]:
    js = f"fetch({json.dumps(url)}, {{credentials: 'include'}}).then(r => r.text())"
    text = await tab.evaluate(js, await_promise=True)
    data = json.loads(text)

    employer_names.append(data["employerName"])
    easy_apply.append(data["isEasyApply"])
    descriptions.append(BeautifulSoup(data["jobDescription"], "html.parser").get_text(" ", strip=True))

    
    await asyncio.sleep(random.uniform(2, 4))   # pause 2–4 seconds between jobs




In [ ]:
df['employer_name']=employer_names
df['easy_apply']=easy_apply
df['descriptions']=descriptions

In [ ]:
df